# Data Ingestion

In [51]:
import logging
import os
from contextlib import contextmanager

import duckdb

# Configure structured logging (force=True so re-running this cell in an
# already-running kernel doesn't just add duplicate handlers).
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("EcoLens.DataPipeline")

DEFAULT_DB_PATH = (
    "/Users/macbook/Project/personal/EcoLens/services/data-pipeline"
    "/data/historical/ecolens_historical.duckdb"
)
db_path = os.getenv("DUCKDB_PATH", DEFAULT_DB_PATH)


@contextmanager
def duckdb_connection(path: str, read_only: bool = True):
    """Connect, log, and guarantee the close+log happens exactly once --
    defined here so every later cell does `with duckdb_connection(...) as con:`
    instead of re-pasting this same try/except/finally block (which is how
    the notebook ended up with two copies of the same ~40 lines, and the
    same "DuckDB connection closed cleanly." log line, drifting out of
    sync with each other after only one of them got edited).
    """
    resolved_path = os.path.expanduser(path)
    if read_only and not os.path.exists(resolved_path):
        logger.error("DuckDB database file not found at path: %s", resolved_path)
        raise FileNotFoundError(f"Database file not found: {resolved_path}")

    con = None
    try:
        con = duckdb.connect(database=resolved_path, read_only=read_only)
        logger.info(
            "Successfully connected (read_only=%s) to %s", read_only, resolved_path
        )
        yield con
    except duckdb.Error as e:
        logger.exception("DuckDB internal error while connecting to %s", resolved_path)
        raise ConnectionError(f"Failed to connect to DuckDB database: {e}") from e
    finally:
        if con is not None:
            con.close()
            logger.info("DuckDB connection closed cleanly.")


with duckdb_connection(db_path) as con:
    version = con.execute("SELECT version()").fetchone()[0]
    logger.info("DuckDB Version active: %s", version)

    tables_df = con.execute(
        "SELECT table_name FROM duckdb_tables WHERE internal = false"
    ).df()
    print("Tables in database:")
    for name in tables_df["table_name"]:
        print(f" - {name}")


2026-07-26 21:07:55,236 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-26 21:07:55,237 [INFO] EcoLens.DataPipeline: DuckDB Version active: v1.5.5
2026-07-26 21:07:55,250 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


Tables in database:
 - aemo_holidays
 - aemo_nem_dispatch
 - aemo_wem_dispatch
 - bom_observations
 - openelectricity_responses


In [52]:
"""
This query generates a complete, continuous daily timeline between a given 
start and end date, and counts how many records exist for each day in the 
'bom_observations' table. If a specific day has no records, it returns 0 
instead of skipping the date entirely.
"""
START = "2025-08-01 00:00:00"
END = "2026-01-02 00:00:00"
table_name = 'openelectricity_responses'

# table_name goes into an f-string, not a `?` placeholder -- DuckDB's
# prepared-statement params can only bind values, not identifiers like
# table/column names, so `FROM ?` bound to "aemo_nem_dispatch" would
# try to match a literal string, not a table. f-string interpolation
# is fine here since table_name is an internal constant, never user input.
query = f"""
   WITH date_spine AS (
    SELECT UNNEST(generate_series(
        CAST(? AS DATE), 
        CAST(? AS DATE) - INTERVAL 1 DAY, 
        INTERVAL 1 DAY
    ))::date AS data_date
),
aggregated_data AS (
    SELECT
        CAST(ts AS DATE) AS data_date,
        COUNT(*) AS record_frequency
    FROM {table_name}
    WHERE ts >= ?
      AND ts <  ?
    GROUP BY CAST(ts AS DATE)
)
SELECT 
    ds.data_date,
    COALESCE(ad.record_frequency, 0) AS record_frequency
FROM date_spine ds
LEFT JOIN aggregated_data ad 
    ON ds.data_date = ad.data_date
ORDER BY ds.data_date ASC;
"""
# 4 `?` placeholders above (date_spine's start/end, then
# aggregated_data's start/end) -- each needs its own value, so START/END
# must be repeated, not just passed once.
query_params = [START, END, START, END]

with duckdb_connection(db_path) as con:
    df = con.execute(query, query_params).df()

df


2026-07-26 21:07:55,594 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb


2026-07-26 21:07:55,615 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


,data_date,record_frequency
0,2025-08-01,576
1,2025-08-02,576
2,2025-08-03,576
3,2025-08-04,576
4,2025-08-05,576
...,...,...
149,2025-12-28,576
150,2025-12-29,576
151,2025-12-30,576
152,2025-12-31,576


In [53]:
# """
# This query generates a complete, continuous daily timeline between a given 
# start and end date, counts how many records exist for each day in the 
# 'aemo_nem_dispatch' table (returning 0 for days with no data), and filters 
# the final results to only show days where the record count is less than X.
# """
# START = "2025-08-01 00:00:00"
# END = "2025-09-01 00:00:00"
# AMOUNT = 1728
# table_name='aemo_nem_dispatch'

# query = f"""
#    WITH date_spine AS (
#     SELECT UNNEST(generate_series(
#         CAST(? AS DATE), 
#         CAST(? AS DATE) - INTERVAL 1 DAY, 
#         INTERVAL 1 DAY
#     ))::date AS data_date
# ),
# aggregated_data AS (
#     SELECT
#         CAST(ts AS DATE) AS data_date,
#         COUNT(*) AS record_frequency
#     FROM {table_name}
#     WHERE ts >= ?
#       AND ts <  ?
#     GROUP BY CAST(ts AS DATE)
# )
# SELECT 
#     ds.data_date,
#     COALESCE(ad.record_frequency, 0) AS record_frequency
# FROM date_spine ds
# LEFT JOIN aggregated_data ad 
#     ON ds.data_date = ad.data_date
# WHERE COALESCE(ad.record_frequency, 0) < ?
# ORDER BY ds.data_date ASC;
# """
# query_params = [START, END, START, END, AMOUNT]

# with duckdb_connection(db_path) as con:
#     df = con.execute(query,query_params).df()

# df


In [54]:
# query ="""
# select * from aemo_holidays limit 1;
# """
# query_params = []


In [55]:
# """
# This query fetch a single row from `bom_observations`
# """

START = "2025-08-01 00:00:00"
END = "2026-01-01 00:00:00"
table_name='aemo_wem_dispatch'
query =f"""
select count(*)/(30*288) as num_month from {table_name}  limit 1;
"""



with duckdb_connection(db_path) as con:
    df = con.execute(query).df()

df


2026-07-26 21:07:57,538 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-26 21:07:57,543 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


,num_month
0,11.766667


# Warehousing

## Data preprocessing